# Semantic Question Deduplication Demo

This notebook demonstrates an end-to-end prototype for semantic deduplication of survey questions using sentence embeddings and approximate nearest neighbor search. The goal is to illustrate practical challenges such as noisy embeddings and threshold sensitivity.

In [ ]:

# Install dependencies if needed
# !pip install sentence-transformers faiss-cpu pandas numpy scikit-learn


In [ ]:

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


## Load Example Survey Questions

In [ ]:

data = {
    "question_id": ["Q1", "Q2", "Q3", "Q4"],
    "question_text": [
        "Apa penghasilan utama rumah tangga Anda?",
        "Berapa pendapatan utama keluarga Anda?",
        "Berapa jumlah anggota rumah tangga?",
        "Jumlah anggota keluarga yang tinggal serumah?"
    ]
}

df = pd.DataFrame(data)
df


## Generate Sentence Embeddings

In [ ]:

model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
embeddings = model.encode(df["question_text"].tolist(), normalize_embeddings=True)

embeddings.shape


## Pairwise Similarity Matrix

In [ ]:

sim_matrix = cosine_similarity(embeddings)
pd.DataFrame(sim_matrix, index=df.question_id, columns=df.question_id)


## Threshold Sensitivity Analysis

In [ ]:

thresholds = [0.6, 0.7, 0.8, 0.9]

results = {}
for t in thresholds:
    pairs = []
    for i in range(len(df)):
        for j in range(i+1, len(df)):
            if sim_matrix[i, j] >= t:
                pairs.append((df.question_id[i], df.question_id[j], sim_matrix[i, j]))
    results[t] = pairs

results



## Observation

Small changes in similarity thresholds can significantly alter which question pairs are considered duplicates.
This illustrates instability caused by noisy embeddings and motivates the need for optimization-based or graph-structured formulations.
